# Build Evaluation Ground Truth using Weighted RRF

**Method:** Weighted Reciprocal Rank Fusion (Weighted RRF)

**Purpose:** Create consensus ground truth from 7 methods using weighted fusion to balance method groups.

**Pipeline:**
1. Load predictions from all 7 methods (top-100)
2. Apply group-based weights to balance contribution:
   - Lexical group (4 methods): 0.25 each
   - Embedding group (2 methods): 0.5 each
   - Fusion group (1 method): 1.0
3. Use Weighted RRF to create consensus ranking
4. Select top-20 as Ground Truth
5. Assign graded relevance by rank position (1-3→rel=3, 4-10→rel=2, 11-20→rel=1)
6. Export single GT file for all methods

**Key Parameters:**
- Methods: 7 retrieval methods
- Queries: 200 queries
- Top-R per method: 100 items
- GT size: 20 items per query
- k_rrf: 100
- Group weights for balanced contribution

In [49]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict

# Configuration
RANDOM_SEED = 41
np.random.seed(RANDOM_SEED)

# Weighted RRF Parameters
TOP_R = 100         # Use top-100 from each method for fusion
GT_SIZE = 20        # Ground truth size per query
RRF_K = 100         # RRF smoothing parameter (k_rrf = 100)

# Methods and their group-based weights
# Total weight per group = 1.0 to prevent "majority group" dominance
METHOD_WEIGHTS = {
    # Lexical group (4 methods): 1.0 total → 0.25 each
    "TFIDF": 0.25,
    "Ingredient_TFIDF": 0.25,
    "Keyword": 0.25,
    "Hybrid": 0.25,
    
    # Embedding group (2 methods): 1.0 total → 0.5 each
    "SBERT_FAISS": 0.5,
    "Hybrid_TFIDF_SBERT": 0.5,
    
    # Fusion group (1 method): 1.0 total → 1.0
    "RARec_Late_Fusion": 1.0
}

METHOD_NAMES = list(METHOD_WEIGHTS.keys())

print(f"Configuration loaded:")
print(f"  Methods: {len(METHOD_NAMES)}")
print(f"  Top-R for fusion: {TOP_R}")
print(f"  GT size: {GT_SIZE}")
print(f"  RRF k: {RRF_K}")
print(f"\nMethod weights:")
for method, weight in METHOD_WEIGHTS.items():
    print(f"  {method:25s}: {weight:.2f}")

Configuration loaded:
  Methods: 7
  Top-R for fusion: 100
  GT size: 20
  RRF k: 100

Method weights:
  TFIDF                    : 0.25
  Ingredient_TFIDF         : 0.25
  Keyword                  : 0.25
  Hybrid                   : 0.25
  SBERT_FAISS              : 0.50
  Hybrid_TFIDF_SBERT       : 0.50
  RARec_Late_Fusion        : 1.00


## 2. Load Predictions from All Methods

In [50]:
# Setup paths
PREDICTIONS_DIR = Path(r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl")
OUTPUT_DIR = Path(r"E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation")

print(f"Predictions directory: {PREDICTIONS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nChecking prediction files...")

# Check which prediction files exist
available_methods = []
for method in METHOD_NAMES:
    pred_file = PREDICTIONS_DIR / f"{method}_pred.jsonl"
    if pred_file.exists():
        available_methods.append(method)
        print(f"  ✓ {method}")
    else:
        print(f"  ✗ {method} - FILE NOT FOUND")

print(f"\nAvailable methods: {len(available_methods)}/{len(METHOD_NAMES)}")

Predictions directory: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\evaluation_jsonl
Output directory: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation

Checking prediction files...
  ✓ TFIDF
  ✓ Ingredient_TFIDF
  ✓ Keyword
  ✓ Hybrid
  ✓ SBERT_FAISS
  ✓ Hybrid_TFIDF_SBERT
  ✓ RARec_Late_Fusion

Available methods: 7/7


In [51]:
def load_pred_rankings(jsonl_path: Path, list_field: str = "relevant_docs", 
                       docid_field: str = "doc_id", max_rank: int = 100) -> Dict[int, List[int]]:
    """
    Load prediction rankings from JSONL file.
    
    Args:
        jsonl_path: Path to prediction JSONL file
        list_field: Field name containing the list of docs (default: "relevant_docs")
        docid_field: Field name for document ID (default: "doc_id")
        max_rank: Maximum number of top items to keep (default: 100)
    
    Returns:
        Dict mapping query_id to list of doc_ids (ordered by score desc)
    """
    rankings = {}
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            qid = int(obj['query_id'])
            items = obj[list_field]
            # Extract doc_ids in order (already sorted by score desc)
            doc_ids = [int(item[docid_field]) for item in items][:max_rank]
            rankings[qid] = doc_ids
    
    return rankings

print("Defined function: load_pred_rankings")

Defined function: load_pred_rankings


In [52]:
# Load all predictions
preds_by_method = {}

print("Loading predictions from all methods...")
print("=" * 80)

for method in available_methods:
    pred_file = PREDICTIONS_DIR / f"{method}_pred.jsonl"
    rankings = load_pred_rankings(pred_file, max_rank=TOP_R)
    preds_by_method[method] = rankings
    
    # Statistics
    num_queries = len(rankings)
    avg_docs = np.mean([len(docs) for docs in rankings.values()])
    
    print(f"{method:25s}: {num_queries:3d} queries, avg {avg_docs:.1f} docs/query")

print("=" * 80)
print(f"\nTotal methods loaded: {len(preds_by_method)}")

# Get common query IDs across all methods
all_query_sets = [set(rankings.keys()) for rankings in preds_by_method.values()]
common_query_ids = sorted(list(set.intersection(*all_query_sets)))

print(f"Common queries across all methods: {len(common_query_ids)}")
print(f"Query ID range: {min(common_query_ids)} - {max(common_query_ids)}")

Loading predictions from all methods...
TFIDF                    : 200 queries, avg 100.0 docs/query
Ingredient_TFIDF         : 200 queries, avg 100.0 docs/query
Keyword                  : 200 queries, avg 100.0 docs/query
Hybrid                   : 200 queries, avg 100.0 docs/query
SBERT_FAISS              : 200 queries, avg 100.0 docs/query
Hybrid_TFIDF_SBERT       : 200 queries, avg 100.0 docs/query
RARec_Late_Fusion        : 200 queries, avg 100.0 docs/query

Total methods loaded: 7
Common queries across all methods: 200
Query ID range: 49 - 10248


## 3. Weighted Reciprocal Rank Fusion (Weighted RRF) Functions

In [53]:
def weighted_rrf_fuse_for_query(preds_by_method: Dict[str, Dict[int, List[int]]], 
                                method_weights: Dict[str, float],
                                query_id: int, 
                                topR: int = 100, 
                                k: int = 100) -> List[Tuple[int, float]]:
    """
    Fuse rankings from multiple methods using Weighted Reciprocal Rank Fusion (Weighted RRF).
    
    Weighted RRF formula: score(d) = sum over methods j of: w_j * 1/(k + rank_j(d))
    
    Args:
        preds_by_method: Dict[method_name] -> Dict[query_id] -> List[doc_id]
        method_weights: Dict[method_name] -> weight value
        query_id: Query ID to fuse
        topR: Use top-R items from each method (default: 100)
        k: RRF smoothing parameter (default: 100)
    
    Returns:
        List of (doc_id, weighted_rrf_score) sorted by score desc, then doc_id asc
    """
    scores = defaultdict(float)
    
    for method, qmap in preds_by_method.items():
        weight = method_weights.get(method, 1.0)
        
        ranking = qmap.get(query_id, [])
        ranking = ranking[:topR]
        
        # Add weighted RRF score contribution from this method
        for rank_idx, doc_id in enumerate(ranking, start=1):  # 1-based rank
            scores[doc_id] += weight * (1.0 / (k + rank_idx))
    
    # Sort by: score desc, then doc_id asc (for deterministic results)
    fused = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    
    return fused


def assign_rel_by_rank(rank_1based: int) -> int:
    """
    Assign graded relevance based on rank position (rank-based labeling).
    
    - Rank 1-3: 3 (highly relevant)
    - Rank 4-10: 2 (relevant)
    - Rank 11-20: 1 (somewhat relevant)
    - Rank > 20: 0 (not relevant)
    
    Args:
        rank_1based: Position in ranked list (1-indexed)
    
    Returns:
        Relevance grade (0-3)
    """
    if rank_1based <= 3:
        return 3
    elif rank_1based <= 10:
        return 2
    elif rank_1based <= 20:
        return 1
    else:
        return 0


print("Defined functions: weighted_rrf_fuse_for_query, assign_rel_by_rank")

Defined functions: weighted_rrf_fuse_for_query, assign_rel_by_rank


In [54]:
# Test Weighted RRF fusion on a sample query
test_qid = common_query_ids[0]

print(f"Testing Weighted RRF fusion...")
print(f"Query ID: {test_qid}")
print(f"Using all 7 methods with group-based weights")
print("=" * 80)

fused_result = weighted_rrf_fuse_for_query(preds_by_method, METHOD_WEIGHTS, test_qid, 
                                            topR=TOP_R, k=RRF_K)

print(f"\nTop 10 fused results:")
print(f"{'Rank':<6} {'Doc ID':<10} {'Weighted RRF Score':<20} {'Rel Grade'}")
print("-" * 60)

for rank, (doc_id, score) in enumerate(fused_result[:10], 1):
    rel = assign_rel_by_rank(rank)
    print(f"{rank:<6} {doc_id:<10} {score:<20.6f} {rel}")

print(f"\nTotal unique documents in pool: {len(fused_result)}")
print(f"Maximum possible pool size: {len(METHOD_NAMES) * TOP_R} = {len(METHOD_NAMES)}×{TOP_R}")

Testing Weighted RRF fusion...
Query ID: 49
Using all 7 methods with group-based weights

Top 10 fused results:
Rank   Doc ID     Weighted RRF Score   Rel Grade
------------------------------------------------------------
1      69         0.029464             3
2      9042       0.029087             3
3      9221       0.028730             3
4      8863       0.026423             2
5      8795       0.025491             2
6      9651       0.025123             2
7      9043       0.025005             2
8      782        0.023785             2
9      8784       0.023054             2
10     8911       0.022884             2

Total unique documents in pool: 320
Maximum possible pool size: 700 = 7×100


In [55]:
## 4. Build Single Consensus Ground Truth for All Methods

In [56]:
def build_consensus_ground_truth(preds_by_method: Dict[str, Dict[int, List[int]]], 
                                 method_weights: Dict[str, float],
                                 query_ids: List[int], 
                                 gt_size: int = 20, 
                                 topR: int = 100, 
                                 k: int = 100,
                                 out_path: Path = None) -> List[dict]:
    """
    Build consensus ground truth for ALL methods using Weighted RRF.
    Fuses rankings from ALL 7 methods with group-based weights.
    
    Args:
        preds_by_method: All method predictions
        method_weights: Weight for each method
        query_ids: List of query IDs to process
        gt_size: Number of items in ground truth per query
        topR: Use top-R from each method for fusion
        k: RRF smoothing parameter
        out_path: Optional path to save JSONL file
    
    Returns:
        List of ground truth entries
    """
    gt_data = []
    
    for qid in query_ids:
        # Fuse rankings from ALL methods using weighted RRF
        fused = weighted_rrf_fuse_for_query(preds_by_method, method_weights, qid, 
                                             topR=topR, k=k)
        
        # Take top GT_SIZE
        fused_top = fused[:gt_size]
        
        # Build ground truth items with relevance grades
        gt_items = []
        for rank, (doc_id, score) in enumerate(fused_top, start=1):
            gt_items.append({
                'doc_id': int(doc_id),
                'score': float(score),
                'rel': int(assign_rel_by_rank(rank))
            })
        
        gt_data.append({
            'query_id': int(qid),
            'top_k': gt_items
        })
    
    # Optionally save to file
    if out_path:
        with open(out_path, 'w', encoding='utf-8') as f:
            for entry in gt_data:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
        print(f"  ✓ Saved: {out_path.name}")
    
    return gt_data


print("Defined function: build_consensus_ground_truth")

Defined function: build_consensus_ground_truth


In [57]:
# Build single consensus ground truth for ALL methods
print("=" * 80)
print("BUILDING CONSENSUS GROUND TRUTH USING WEIGHTED RRF")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  Total methods: {len(available_methods)}")
print(f"  Total queries: {len(common_query_ids)}")
print(f"  GT size per query: {GT_SIZE}")
print(f"  Weighted RRF parameters: topR={TOP_R}, k={RRF_K}")
print(f"\nMethod weights:")
for method in available_methods:
    weight = METHOD_WEIGHTS[method]
    print(f"  {method:25s}: {weight:.2f}")

print(f"\n{'='*80}")
print("Building single GT file...")
print("-" * 80)

# Output path for single GT file
out_file = OUTPUT_DIR / "gt_consensus_weighted_rrf.jsonl"

# Build consensus GT
consensus_gt = build_consensus_ground_truth(
    preds_by_method=preds_by_method,
    method_weights=METHOD_WEIGHTS,
    query_ids=common_query_ids,
    gt_size=GT_SIZE,
    topR=TOP_R,
    k=RRF_K,
    out_path=out_file
)

print("-" * 80)
print(f"✓ Created consensus ground truth file: {out_file.name}")
print(f"  - Queries: {len(consensus_gt)}")
print(f"  - Items per query: {GT_SIZE}")
print(f"  - Total judgments: {len(consensus_gt) * GT_SIZE}")
print("=" * 80)

BUILDING CONSENSUS GROUND TRUTH USING WEIGHTED RRF

Configuration:
  Total methods: 7
  Total queries: 200
  GT size per query: 20
  Weighted RRF parameters: topR=100, k=100

Method weights:
  TFIDF                    : 0.25
  Ingredient_TFIDF         : 0.25
  Keyword                  : 0.25
  Hybrid                   : 0.25
  SBERT_FAISS              : 0.50
  Hybrid_TFIDF_SBERT       : 0.50
  RARec_Late_Fusion        : 1.00

Building single GT file...
--------------------------------------------------------------------------------
  ✓ Saved: gt_consensus_weighted_rrf.jsonl
--------------------------------------------------------------------------------
✓ Created consensus ground truth file: gt_consensus_weighted_rrf.jsonl
  - Queries: 200
  - Items per query: 20
  - Total judgments: 4000


In [58]:
## 5. Inspect Sample Ground Truth

In [59]:
# Inspect first ground truth entry
sample_gt = consensus_gt[0]

print(f"Sample Ground Truth Entry")
print("=" * 80)
print(f"Query ID: {sample_gt['query_id']}")
print(f"Ground truth items: {len(sample_gt['top_k'])}")
print(f"\nAll {GT_SIZE} items:")
print(f"{'Rank':<6} {'Doc ID':<10} {'Weighted RRF Score':<20} {'Rel Grade'}")
print("-" * 60)

for rank, item in enumerate(sample_gt['top_k'], 1):
    print(f"{rank:<6} {item['doc_id']:<10} {item['score']:<20.6f} {item['rel']}")

print("\n" + "=" * 80)
print(f"Relevance distribution in this GT:")
rel_counts = pd.Series([item['rel'] for item in sample_gt['top_k']]).value_counts().sort_index()
for rel, count in rel_counts.items():
    rel_label = {3: "Highly relevant", 2: "Relevant", 1: "Somewhat relevant"}
    print(f"  Grade {rel} ({rel_label.get(rel, 'Unknown')}): {count} items")
print("=" * 80)

Sample Ground Truth Entry
Query ID: 49
Ground truth items: 20

All 20 items:
Rank   Doc ID     Weighted RRF Score   Rel Grade
------------------------------------------------------------
1      69         0.029464             3
2      9042       0.029087             3
3      9221       0.028730             3
4      8863       0.026423             2
5      8795       0.025491             2
6      9651       0.025123             2
7      9043       0.025005             2
8      782        0.023785             2
9      8784       0.023054             2
10     8911       0.022884             2
11     9705       0.022114             1
12     70         0.022049             1
13     8929       0.021545             1
14     8805       0.020711             1
15     9311       0.019514             1
16     39         0.019164             1
17     9199       0.019008             1
18     9141       0.018236             1
19     1185       0.018139             1
20     9300       0.017869        

In [60]:
## 6. Analyze Pool Coverage and Method Contribution

In [61]:
# Analyze pool coverage for sample query
sample_qid = common_query_ids[0]

print(f"Pool Analysis for Query {sample_qid}")
print("=" * 80)

# Get predictions from each method
print(f"\nMethod contributions (top-{TOP_R} candidates each):")
print(f"{'Method':<25} {'Weight':<10} {'Candidates':<12} {'In GT@{GT_SIZE}'}")
print("-" * 80)

gt_docs = set(item['doc_id'] for item in consensus_gt[0]['top_k'])

for method in available_methods:
    weight = METHOD_WEIGHTS[method]
    candidates = preds_by_method[method].get(sample_qid, [])[:TOP_R]
    in_gt = sum(1 for doc_id in candidates if doc_id in gt_docs)
    print(f"{method:<25} {weight:<10.2f} {len(candidates):<12} {in_gt}")

# Calculate pool statistics
all_candidates = set()
for method in available_methods:
    candidates = preds_by_method[method].get(sample_qid, [])[:TOP_R]
    all_candidates.update(candidates)

print(f"\n{'='*80}")
print(f"Pool statistics:")
print(f"  Total unique candidates: {len(all_candidates)}")
print(f"  Maximum possible: {len(available_methods) * TOP_R}")
print(f"  Overlap ratio: {1 - len(all_candidates)/(len(available_methods) * TOP_R):.2%}")
print(f"  Final GT size: {len(gt_docs)}")
print("=" * 80)

Pool Analysis for Query 49

Method contributions (top-100 candidates each):
Method                    Weight     Candidates   In GT@{GT_SIZE}
--------------------------------------------------------------------------------
TFIDF                     0.25       100          16
Ingredient_TFIDF          0.25       100          16
Keyword                   0.25       100          12
Hybrid                    0.25       100          16
SBERT_FAISS               0.50       100          20
Hybrid_TFIDF_SBERT        0.50       100          20
RARec_Late_Fusion         1.00       100          20

Pool statistics:
  Total unique candidates: 320
  Maximum possible: 700
  Overlap ratio: 54.29%
  Final GT size: 20


## 7. Summary Statistics

In [62]:
# Calculate comprehensive statistics
all_rrf_scores = []
all_rel_grades = []

for query_entry in consensus_gt:
    for item in query_entry['top_k']:
        all_rrf_scores.append(item['score'])
        all_rel_grades.append(item['rel'])

print("=" * 80)
print("CONSENSUS GROUND TRUTH STATISTICS (WEIGHTED RRF)")
print("=" * 80)

print(f"\nDataset Configuration:")
print(f"  Methods: {len(available_methods)}")
print(f"  Queries: {len(consensus_gt)}")
print(f"  GT size per query: {GT_SIZE}")
print(f"  Total judgments: {len(all_rrf_scores)}")
print(f"  Top-R per method: {TOP_R}")

print(f"\nWeighted RRF Score Statistics:")
print(f"  Mean:   {np.mean(all_rrf_scores):.6f}")
print(f"  Median: {np.median(all_rrf_scores):.6f}")
print(f"  Std:    {np.std(all_rrf_scores):.6f}")
print(f"  Min:    {np.min(all_rrf_scores):.6f}")
print(f"  Max:    {np.max(all_rrf_scores):.6f}")

print(f"\nRelevance Grade Distribution (Overall):")
grade_counts = pd.Series(all_rel_grades).value_counts().sort_index()
grade_labels = {3: "Highly relevant (rank 1-3)", 
                2: "Relevant (rank 4-10)", 
                1: "Somewhat relevant (rank 11-20)"}
total = len(all_rel_grades)
for grade, count in grade_counts.items():
    percentage = count / total * 100
    label = grade_labels.get(grade, "Unknown")
    print(f"  Grade {grade} ({label}): {count:5d} ({percentage:5.1f}%)")

print(f"\nExpected distribution (for {len(consensus_gt)} queries):")
print(f"  Grade 3: {len(consensus_gt) * 3} ({3/20*100:.1f}% of GT)")
print(f"  Grade 2: {len(consensus_gt) * 7} ({7/20*100:.1f}% of GT)")
print(f"  Grade 1: {len(consensus_gt) * 10} ({10/20*100:.1f}% of GT)")

print(f"\nMethod Configuration:")
print(f"  ✓ Weighted Reciprocal Rank Fusion (Weighted RRF)")
print(f"  ✓ Group-balanced weights (Lexical:Embedding:Fusion = 1:1:1)")
print(f"  ✓ k_rrf = {RRF_K}")
print(f"  ✓ Rank-based relevance grading")
print(f"  ✓ Deterministic sorting (score desc, doc_id asc)")

print("=" * 80)

CONSENSUS GROUND TRUTH STATISTICS (WEIGHTED RRF)

Dataset Configuration:
  Methods: 7
  Queries: 200
  GT size per query: 20
  Total judgments: 4000
  Top-R per method: 100

Weighted RRF Score Statistics:
  Mean:   0.017659
  Median: 0.017090
  Std:    0.003870
  Min:    0.009692
  Max:    0.029679

Relevance Grade Distribution (Overall):
  Grade 1 (Somewhat relevant (rank 11-20)):  2000 ( 50.0%)
  Grade 2 (Relevant (rank 4-10)):  1400 ( 35.0%)
  Grade 3 (Highly relevant (rank 1-3)):   600 ( 15.0%)

Expected distribution (for 200 queries):
  Grade 3: 600 (15.0% of GT)
  Grade 2: 1400 (35.0% of GT)
  Grade 1: 2000 (50.0% of GT)

Method Configuration:
  ✓ Weighted Reciprocal Rank Fusion (Weighted RRF)
  ✓ Group-balanced weights (Lexical:Embedding:Fusion = 1:1:1)
  ✓ k_rrf = 100
  ✓ Rank-based relevance grading
  ✓ Deterministic sorting (score desc, doc_id asc)


In [63]:
# Analyze score distribution across queries
print("\nWeighted RRF Score Statistics by Relevance Grade:")
print("=" * 80)

for grade in sorted(pd.Series(all_rel_grades).unique(), reverse=True):
    scores_for_grade = [score for score, rel in zip(all_rrf_scores, all_rel_grades) if rel == grade]
    if scores_for_grade:
        grade_label = {3: "Highly relevant", 2: "Relevant", 1: "Somewhat relevant"}
        print(f"\nGrade {grade} ({grade_label.get(grade, 'Unknown')}):")
        print(f"  Count:  {len(scores_for_grade)}")
        print(f"  Mean:   {np.mean(scores_for_grade):.6f}")
        print(f"  Median: {np.median(scores_for_grade):.6f}")
        print(f"  Min:    {np.min(scores_for_grade):.6f}")
        print(f"  Max:    {np.max(scores_for_grade):.6f}")

print("\n" + "=" * 80)
print(f"Note: All 7 methods contribute to this single consensus GT")
print(f"Group weights ensure balanced contribution: Lexical=1.0, Embedding=1.0, Fusion=1.0")
print("=" * 80)


Weighted RRF Score Statistics by Relevance Grade:

Grade 3 (Highly relevant):
  Count:  600
  Mean:   0.022838
  Median: 0.023012
  Min:    0.013235
  Max:    0.029679

Grade 2 (Relevant):
  Count:  1400
  Mean:   0.018623
  Median: 0.018478
  Min:    0.010524
  Max:    0.027604

Grade 1 (Somewhat relevant):
  Count:  2000
  Mean:   0.015430
  Median: 0.015287
  Min:    0.009692
  Max:    0.024654

Note: All 7 methods contribute to this single consensus GT
Group weights ensure balanced contribution: Lexical=1.0, Embedding=1.0, Fusion=1.0


## 8. Output Summary & Next Steps

In [64]:
print("\n" + "=" * 80)
print("✓ CONSENSUS GROUND TRUTH GENERATION COMPLETED")
print("=" * 80)

gt_filename = "gt_consensus_weighted_rrf.jsonl"
gt_filepath = OUTPUT_DIR / gt_filename
file_size = gt_filepath.stat().st_size / 1024  # KB

print(f"\nGenerated File:")
print(f"  ✓ {gt_filename:<40} ({file_size:.1f} KB)")
print(f"  Path: {gt_filepath}")

print(f"\nGround Truth Details:")
print(f"  Queries: {len(consensus_gt)}")
print(f"  Items per query: {GT_SIZE}")
print(f"  Total judgments: {len(consensus_gt) * GT_SIZE}")

print(f"\nFile Format:")
print(f"  Each line: {{'query_id': int, 'top_k': [list of {GT_SIZE} items]}}")
print(f"  Each item: {{'doc_id': int, 'score': float, 'rel': int (0-3)}}")

print(f"\nRelevance Grades:")
print(f"  3 = Highly relevant (rank 1-3)")
print(f"  2 = Relevant (rank 4-10)")
print(f"  1 = Somewhat relevant (rank 11-20)")

print(f"\n{'='*80}")
print("NEXT STEPS:")
print("=" * 80)
print(f"1. Use this single GT file to evaluate ALL 7 methods")
print(f"2. For each method M:")
print(f"   - Load predictions: <M>_pred.jsonl")
print(f"   - Load ground truth: {gt_filename}")
print(f"   - Calculate metrics: Precision@10, Recall@10, MRR@10, nDCG@10, MAP@10")
print(f"3. Compare results across all 7 methods")
print(f"4. All methods evaluated against same consensus GT")

print(f"\nKey Advantages of Weighted RRF:")
print(f"  ✓ Balanced contribution from all method groups")
print(f"  ✓ No single group dominates (Lexical:Embedding:Fusion = 1:1:1)")
print(f"  ✓ Fair consensus across different approaches")
print(f"  ✓ Stable rankings with topR=100, k_rrf=100")
print(f"  ✓ Deterministic results (tie-breaking by doc_id)")
print(f"  ✓ Single GT for all methods (consistent evaluation)")

print("=" * 80)
print(f"\nPipeline Configuration Summary:")
print(f"  Methods: {len(METHOD_NAMES)}")
print(f"  Candidates per method: {TOP_R}")
print(f"  RRF smoothing (k): {RRF_K}")
print(f"  GT size: {GT_SIZE}")
print(f"  Weighting scheme: Group-balanced")
for i, (method, weight) in enumerate(METHOD_WEIGHTS.items(), 1):
    print(f"    {i}. {method:25s} = {weight:.2f}")
print("=" * 80)


✓ CONSENSUS GROUND TRUTH GENERATION COMPLETED

Generated File:
  ✓ gt_consensus_weighted_rrf.jsonl          (235.1 KB)
  Path: E:\DS300-UIT-RecommenderSystem\Finalproject\notebooks\recommend_and_evaluation\gt_consensus_weighted_rrf.jsonl

Ground Truth Details:
  Queries: 200
  Items per query: 20
  Total judgments: 4000

File Format:
  Each line: {'query_id': int, 'top_k': [list of 20 items]}
  Each item: {'doc_id': int, 'score': float, 'rel': int (0-3)}

Relevance Grades:
  3 = Highly relevant (rank 1-3)
  2 = Relevant (rank 4-10)
  1 = Somewhat relevant (rank 11-20)

NEXT STEPS:
1. Use this single GT file to evaluate ALL 7 methods
2. For each method M:
   - Load predictions: <M>_pred.jsonl
   - Load ground truth: gt_consensus_weighted_rrf.jsonl
   - Calculate metrics: Precision@10, Recall@10, MRR@10, nDCG@10, MAP@10
3. Compare results across all 7 methods
4. All methods evaluated against same consensus GT

Key Advantages of Weighted RRF:
  ✓ Balanced contribution from all method gro